**Full Data Science Pipeline**  
- Stage 1: Data Loading  
- Stage 2: Exploratory Data Analysis (EDA)  
- Stage 3: Preprocessing & Feature Engineering  
- Stage 4: Model Training & Evaluation  
- Stage 5: Model Saving  

**Dataset**: Telco Customer Churn (Kaggle)  
**Goal**: Predict customer churn (Yes/No)  
**Best Model**: Logistic Regression (F1=0.6084, AUC=0.8346)


## STAGE 1 : DATA LOADING & INITIAL INSPECTION

In [ ]:
# Install required libraries
"""pip install pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn shap openpyxl"""

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import io
import warnings
warnings.filterwarnings('ignore')

# Display settings
plt.style.use('ggplot')
sns.set_palette("Set2")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print(" All libraries have been successfully installed and loaded.")

# Importing Data File (from colab)
print(" Please upload the file: WA_Fn-UseC_-Telco-Customer-Churn.csv")
uploaded = files.upload()



# Load the uploaded file
df = pd.read_csv(io.BytesIO(uploaded['WA_Fn-UseC_-Telco-Customer-Churn.csv']))

# 1. Show first 5 rows
print(" First 5 rows of the data:")
display(df.head())

# 2. General information about the data
print("\n General information about the data:")
print(df.info())

# 3. Descriptive statistics
print("\n Descriptive statistics for numerical columns:")
display(df.describe())

# 4. Check target variable distribution
print("\n Target variable (Churn) distribution:")
print(df['Churn'].value_counts())
print("\nPercentage:")
print(df['Churn'].value_counts(normalize=True) * 100)

# 5. Check for missing values
print("\n Missing values in each column:")
print(df.isnull().sum())

# Descriptive statistics for numerical columns
print(" Descriptive statistics for numerical variables:")
display(df.describe())

# Statistics for categorical variables
print("\n Descriptive statistics for categorical variables:")
display(df.describe(include=['object']))

# Plot histograms for numerical variables
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(numerical_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Select key categorical variables
categorical_key = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Contract', 'PaymentMethod']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_key):
    sns.countplot(x=col, data=df, ax=axes[i], hue=col, palette='Set2', legend=False)
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    # Rotate labels if needed
    if col == 'PaymentMethod':
        axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Create a function to compute churn rate by feature
def churn_rate_by_feature(feature):
    churn_rate = df.groupby(feature)['Churn'].value_counts(normalize=True).unstack() * 100
    return churn_rate

# Plot churn rate for key features
features_to_plot = ['Contract', 'PaymentMethod', 'InternetService', 'SeniorCitizen']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, feature in enumerate(features_to_plot):
    churn_data = df.groupby(feature)['Churn'].value_counts(normalize=True).unstack() * 100
    churn_data.plot(kind='bar', stacked=True, ax=axes[i], color=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'Churn Rate by {feature}')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Percentage')
    axes[i].legend(['Stayed', 'Churned'], loc='upper right')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))

# Boxplot of Tenure by Churn
sns.boxplot(x='Churn', y='tenure', data=df, palette=['#2ecc71', '#e74c3c'])
plt.title('Tenure (months) for Churned vs Stayed Customers')
plt.xlabel('Customer Status (0=Stayed, 1=Churned)')
plt.ylabel('Tenure (months)')
plt.show()

# Select numerical columns for correlation
numeric_df = df[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']]
# Add Churn as a numeric variable
numeric_df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Numerical Variables')
plt.show()

# Boxplots for outlier detection
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(['tenure', 'MonthlyCharges', 'TotalCharges']):
    sns.boxplot(y=df[col], ax=axes[i], color='lightblue')
    axes[i].set_title(f'Outliers in {col}')
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

# Pairplot of key numerical features with Churn coloring
sample_df = df[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].copy()
# Convert Churn to numeric for color coding
sample_df['Churn'] = sample_df['Churn'].map({'Yes': 1, 'No': 0})

sns.pairplot(sample_df, hue='Churn', diag_kind='kde', palette=['#2ecc71', '#e74c3c'])
plt.suptitle('Pairplot of Numerical Variables by Churn', y=1.02)
plt.show()


## STAGE 2: EXPLORATORY DATA ANALYSIS (EDA)

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('ggplot')
sns.set_palette("Set2")


# 2.1 Descriptive Statistics
print("="*50)
print("DESCRIPTIVE STATISTICS")
print("="*50)

# Numerical features
print("\n Numerical features:")
display(df.describe())

# Categorical features
print("\n Categorical features:")
display(df.describe(include=['object']))


# 2.2 Distribution of Numerical Features
print("\n" + "="*50)
print("DISTRIBUTION OF NUMERICAL FEATURES")
print("="*50)

numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(numerical_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.tight_layout()
plt.show()


# 2.3 Distribution of Key Categorical Features
print("\n" + "="*50)
print("DISTRIBUTION OF CATEGORICAL FEATURES")
print("="*50)

categorical_key = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Contract', 'PaymentMethod']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_key):
    sns.countplot(x=col, data=df, ax=axes[i], hue=col, palette='Set2', legend=False)
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    if col == 'PaymentMethod':
        axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


# 2.4 Churn Rate by Feature (Key Insights)
print("\n" + "="*50)
print("CHURN RATE BY FEATURE")
print("="*50)

features_to_plot = ['Contract', 'PaymentMethod', 'InternetService', 'SeniorCitizen']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, feature in enumerate(features_to_plot):
    churn_data = df.groupby(feature)['Churn'].value_counts(normalize=True).unstack() * 100
    churn_data.plot(kind='bar', stacked=True, ax=axes[i], color=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'Churn Rate by {feature}')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Percentage')
    axes[i].legend(['Stayed', 'Churned'], loc='upper right')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


# 2.5 Tenure vs Churn (Boxplot)
print("\n" + "="*50)
print("TENURE VS CHURN")
print("="*50)

plt.figure(figsize=(12, 6))
sns.boxplot(x='Churn', y='tenure', data=df, palette=['#2ecc71', '#e74c3c'])
plt.title('Tenure Distribution by Churn Status')
plt.xlabel('Churn Status (0=Stayed, 1=Churned)')
plt.ylabel('Tenure (months)')
plt.show()


# 2.6 Correlation Matrix (FIXED VERSION)
print("\n" + "="*50)
print("CORRELATION MATRIX")
print("="*50)

# Create numeric dataframe properly
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
numeric_df = df[numeric_cols].copy()

# Convert TotalCharges to numeric (it may contain strings)
numeric_df['TotalCharges'] = pd.to_numeric(numeric_df['TotalCharges'], errors='coerce')

# Fill NaN values with mean
numeric_df['TotalCharges'].fillna(numeric_df['TotalCharges'].mean(), inplace=True)

# Add Churn as numeric
numeric_df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Verify data types
print("\n Data types:")
print(numeric_df.dtypes)

print("\n First 5 rows:")
print(numeric_df.head())

# Plot heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.show()


# 2.7 Outlier Detection (Boxplots)
print("\n" + "="*50)
print("OUTLIER DETECTION")
print("="*50)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(['tenure', 'MonthlyCharges', 'TotalCharges']):
    sns.boxplot(y=df[col], ax=axes[i], color='lightblue')
    axes[i].set_title(f'Outliers in {col}')
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()


# 2.8 Pairplot of Key Features
print("\n" + "="*50)
print("PAIRPLOT OF KEY FEATURES")
print("="*50)

# Create a copy for pairplot with numeric Churn
pairplot_df = df[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].copy()
pairplot_df['Churn'] = pairplot_df['Churn'].map({'Yes': 1, 'No': 0})

sns.pairplot(pairplot_df, hue='Churn', diag_kind='kde', palette=['#2ecc71', '#e74c3c'])
plt.suptitle('Pairplot of Numerical Features by Churn', y=1.02)
plt.show()


# 2.9 Key Insights Summary
print("\n" + "="*50)
print(" KEY INSIGHTS FROM EDA")
print("="*50)

print("""
1. **Strongest Predictor: Contract Type**
   - Monthly contract: ~42% churn rate
   - One-year contract: ~19% churn rate
   - Two-year contract: ~3% churn rate

2. **Tenure (Customer Lifetime)**
   - Customers with longer tenure (>40 months) have very low churn
   - Customers with short tenure (<10 months) have highest churn

3. **Payment Method**
   - Electronic check has the highest churn rate

4. **Senior Citizen**
   - Senior citizens have lower churn rate

5. **Internet Service**
   - Fiber optic users have higher churn than DSL users

6. **Charges**
   - Higher MonthlyCharges → higher churn
   - Higher TotalCharges (due to longer tenure) → lower churn

7. **Correlation Matrix Insights**
   - Tenure strongly correlates with TotalCharges (0.82)
   - Tenure has negative correlation with Churn (-0.35)
   - MonthlyCharges has positive correlation with Churn (0.19)
""")

print("\n STAGE 2 (EDA) COMPLETED SUCCESSFULLY!")
print("="*50)

## STAGE 3: DATA PREPROCESSING & FEATURE ENGINEERING

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

print("="*60)
print("STAGE 3: DATA PREPROCESSING & FEATURE ENGINEERING")
print("="*60)


# 3.1 Separate Features and Target
print("\n 3.1 Separating Features (X) and Target (y)")
print("-"*40)

X = df.drop('Churn', axis=1)
y = df['Churn']

print(f" Features (X) shape: {X.shape}")
print(f" Target (y) shape: {y.shape}")
print("\nTarget distribution:")
print(y.value_counts(normalize=True) * 100)


# 3.2 Split Data into Train and Test Sets
print("\n 3.2 Splitting Data into Train/Test Sets")
print("-"*40)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f" Training set size: {X_train.shape[0]} samples")
print(f" Test set size: {X_test.shape[0]} samples")
print("\nTarget distribution in training set:")
print(y_train.value_counts(normalize=True) * 100)
print("\nTarget distribution in test set:")
print(y_test.value_counts(normalize=True) * 100)


# 3.3 Identify Numerical and Categorical Columns
print("\n 3.3 Identifying Numerical and Categorical Columns")
print("-"*40)

numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f" Numerical columns ({len(numerical_cols)}):")
print(numerical_cols)
print(f"\n Categorical columns ({len(categorical_cols)}):")
print(categorical_cols)


# 3.4 Build Preprocessing Pipeline
print("\n 3.4 Building Preprocessing Pipeline")
print("-"*40)

# Pipeline for numerical data (Standardization)
numerical_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

# Pipeline for categorical data (One-Hot Encoding)
categorical_pipeline = Pipeline([
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

print(" Preprocessing pipeline created successfully!")
print("\nPipeline structure:")
print(preprocessor)


# 3.5 Apply Preprocessing to Training and Test Data
print("\n 3.5 Applying Preprocessing to Data")
print("-"*40)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f" Training data shape after preprocessing: {X_train_processed.shape}")
print(f" Test data shape after preprocessing: {X_test_processed.shape}")

# Get feature names after transformation
feature_names = preprocessor.get_feature_names_out()
print(f"\n Total number of features after transformation: {len(feature_names)}")
print("\nSample of new feature names (first 10):")
print(feature_names[:10])

# 3.6 Save Preprocessor for Later Use (API / Deployment)
print("\n 3.6 Saving Preprocessor")
print("-"*40)

joblib.dump(preprocessor, 'preprocessor.pkl')
print(" Preprocessor saved as 'preprocessor.pkl'")


# 3.7 Summary Statistics of Preprocessed Data
print("\n 3.7 Summary of Preprocessed Data")
print("-"*40)

print("\n Training data statistics:")
print(f"  - Mean: {np.mean(X_train_processed):.4f}")
print(f"  - Standard deviation: {np.std(X_train_processed):.4f}")
print(f"  - Min value: {np.min(X_train_processed):.4f}")
print(f"  - Max value: {np.max(X_train_processed):.4f}")

print("\n Test data statistics:")
print(f"  - Mean: {np.mean(X_test_processed):.4f}")
print(f"  - Standard deviation: {np.std(X_test_processed):.4f}")
print(f"  - Min value: {np.min(X_test_processed):.4f}")
print(f"  - Max value: {np.max(X_test_processed):.4f}")


# 3.8 Data Types After Preprocessing
print("\n 3.8 Data Types After Preprocessing")
print("-"*40)

print(f"Training data type: {X_train_processed.dtype}")
print(f"Test data type: {X_test_processed.dtype}")


# 3.9 Check for Missing Values After Preprocessing
print("\n 3.9 Checking for Missing Values")
print("-"*40)

print(f"Missing values in training data: {np.isnan(X_train_processed).sum()}")
print(f"Missing values in test data: {np.isnan(X_test_processed).sum()}")


# 3.10 Final Summary
print("\n" + "="*60)
print(" STAGE 3 COMPLETED SUCCESSFULLY!")
print("="*60)
print("\n Summary of Preprocessing:")
print(f"  - Original features: {X.shape[1]}")
print(f"  - Features after preprocessing: {X_train_processed.shape[1]}")
print(f"  - Training samples: {X_train_processed.shape[0]}")
print(f"  - Test samples: {X_test_processed.shape[0]}")
print(f"  - Preprocessor saved: preprocessor.pkl")
print("\n Ready for Stage 4: Model Training & Evaluation!")
print("="*60)

## STAGE 4: MODEL TRAINING & EVALUATION

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("STAGE 4: MODEL TRAINING (FIXED - NO SPEED OPTIMIZATION)")
print("="*60)


print("\n Converting y to numeric (0=No, 1=Yes)...")
if y_train.dtype == 'object' or y_train.dtype.name == 'category':
    y_train = y_train.map({'No': 0, 'Yes': 1})
    y_test = y_test.map({'No': 0, 'Yes': 1})
    print(" y_train and y_test converted to numeric")
else:
    print(" y_train and y_test are already numeric")

#Remove CustomerID for nan F1 Score handling
print("\n Removing 'customerID' from features...")
X_train_clean = X_train.drop('customerID', axis=1)
X_test_clean = X_test.drop('customerID', axis=1)
print(f" Features after removing customerID: {X_train_clean.shape[1]}")


# Rebuild preprocessor without customerID
print("\n Building new preprocessor...")
numerical_cols = X_train_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train_clean.select_dtypes(include=['object']).columns.tolist()

numerical_pipeline = Pipeline([('scaler', StandardScaler())])
categorical_pipeline = Pipeline([
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

X_train_processed = preprocessor.fit_transform(X_train_clean)
X_test_processed = preprocessor.transform(X_test_clean)
print(f" New feature count: {X_train_processed.shape[1]}")


# SMOTE
print("\n Applying SMOTE...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_processed, y_train)
print(f" Original training set shape: {X_train_processed.shape}")
print(f" Resampled training set shape: {X_train_resampled.shape}")
print("\nTarget distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())


# Define models and hyperparameter grids 
print("\n Defining Models and Hyperparameter Grids")
print("-"*40)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
}

param_grids = {
    'Logistic Regression': {
        'C': [0.1, 1.0, 10.0],
        'solver': ['liblinear', 'lbfgs']
    },
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, 15],
        'min_samples_split': [2, 5, 10]
    },
    'XGBoost': {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.8, 1.0]
    }
}

print(" Models and hyperparameter grids defined (full search).")


# Train with GridSearchCV (5-fold, all combinations)
print("\n Training Models with GridSearchCV (5-fold CV)")
print("-"*40)

best_models = {}
results = []

for name, model in models.items():
    print(f"\n Training {name}...")

    grid_search = GridSearchCV(
        model,
        param_grids[name],
        cv=5,
        scoring='f1',
        n_jobs=-1,
        verbose=0
    )

    grid_search.fit(X_train_resampled, y_train_resampled)
    best_models[name] = grid_search.best_estimator_

    cv_scores = cross_val_score(
        grid_search.best_estimator_,
        X_train_resampled,
        y_train_resampled,
        cv=5,
        scoring='f1'
    )

    print(f"    Best parameters: {grid_search.best_params_}")
    print(f"    CV F1-Score: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

    results.append({
        'Model': name,
        'Best Params': grid_search.best_params_,
        'CV F1 Mean': cv_scores.mean(),
        'CV F1 Std': cv_scores.std()
    })


# Evaluate on Test Set
print("\n" + "="*60)
print(" Model Evaluation on Test Set")
print("="*60)

evaluation_results = []

for name, model in best_models.items():
    print(f"\n Evaluating {name} on Test Set:")
    print("-"*40)

    y_pred = model.predict(X_test_processed)
    y_pred_proba = model.predict_proba(X_test_processed)[:, 1]

    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_pred_proba),
        'MCC': matthews_corrcoef(y_test, y_pred),
        'Cohen Kappa': cohen_kappa_score(y_test, y_pred)
    }
    evaluation_results.append(metrics)

    print(f"    Accuracy: {metrics['Accuracy']:.4f}")
    print(f"    Balanced Accuracy: {metrics['Balanced Accuracy']:.4f}")
    print(f"    Precision: {metrics['Precision']:.4f}")
    print(f"    Recall: {metrics['Recall']:.4f}")
    print(f"    F1-Score: {metrics['F1-Score']:.4f}")
    print(f"    AUC-ROC: {metrics['AUC-ROC']:.4f}")
    print(f"    MCC: {metrics['MCC']:.4f}")
    print(f"    Cohen Kappa: {metrics['Cohen Kappa']:.4f}")

    print(f"\n    Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Stayed', 'Churned']))


# Model Comparison
print("\n" + "="*60)
print(" Model Comparison")
print("="*60)

comparison_df = pd.DataFrame(evaluation_results)
comparison_df = comparison_df.round(4)
print("\n Model Performance Comparison:")
display(comparison_df)

best_model_name = comparison_df.loc[comparison_df['F1-Score'].idxmax(), 'Model']
best_model = best_models[best_model_name]
print(f"\n Best Model: {best_model_name} (F1-Score: {comparison_df['F1-Score'].max():.4f})")


# Visualizations (Bar plot, Heatmap, Confusion Matrices, ROC, Feature Importance)
print("\n Visualizations")
print("-"*40)

# Bar plot and Heatmap
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

comparison_df_melted = comparison_df.melt(
    id_vars=['Model'],
    value_vars=metrics_to_plot,
    var_name='Metric',
    value_name='Score'
)
sns.barplot(x='Metric', y='Score', hue='Model', data=comparison_df_melted, ax=axes[0])
axes[0].set_title('Model Performance Comparison')
axes[0].set_ylim(0, 1)
axes[0].legend(loc='lower right')

heatmap_data = comparison_df.set_index('Model')[metrics_to_plot]
sns.heatmap(heatmap_data, annot=True, cmap='YlGnBu', fmt='.4f', ax=axes[1])
axes[1].set_title('Performance Heatmap')
plt.tight_layout()
plt.show()

# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes = axes.flatten()
for i, (name, model) in enumerate(best_models.items()):
    y_pred = model.predict(X_test_processed)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Stayed', 'Churned'],
                yticklabels=['Stayed', 'Churned'])
    axes[i].set_title(f'{name} - Confusion Matrix')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')
plt.tight_layout()
plt.show()

# ROC Curves
plt.figure(figsize=(10, 8))
for name, model in best_models.items():
    y_pred_proba = model.predict_proba(X_test_processed)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

# Feature Importance for Random Forest & XGBoost
feature_names = preprocessor.get_feature_names_out()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes = axes.flatten()
for i, model_name in enumerate(['Random Forest', 'XGBoost']):
    if model_name in best_models:
        model = best_models[model_name]
        if hasattr(model, 'feature_importances_'):
            importances = model.feature_importances_
            indices = np.argsort(importances)[-20:]
            axes[i].barh(range(20), importances[indices])
            axes[i].set_yticks(range(20))
            axes[i].set_yticklabels([feature_names[j] for j in indices])
            axes[i].set_title(f'{model_name} - Top 20 Feature Importances')
            axes[i].set_xlabel('Importance')
plt.tight_layout()
plt.show()


# Save Best Model
print("\n Saving Best Model")
print("-"*40)
joblib.dump(best_model, 'best_churn_model.pkl')
joblib.dump(preprocessor, 'preprocessor.pkl')
print(f" Best model ({best_model_name}) saved as 'best_churn_model.pkl'")
print(" Preprocessor saved as 'preprocessor.pkl'")


# Final Summary
print("\n" + "="*60)
print(" STAGE 4 COMPLETED SUCCESSFULLY!")
print("="*60)
print("\n Final Summary:")
print(f"  - Best Model: {best_model_name}")
print(f"  - F1-Score: {comparison_df['F1-Score'].max():.4f}")
print(f"  - AUC-ROC: {comparison_df['AUC-ROC'].max():.4f}")
print(f"  - Accuracy: {comparison_df['Accuracy'].max():.4f}")
print(f"  - Model saved: best_churn_model.pkl")
print(f"  - Preprocessor saved: preprocessor.pkl")
print("\n Model Performance Summary Table:")
display(comparison_df)
print("\n Ready for Stage 5: Deployment!")
print("="*60)